# Lab: Interrupted Time Series Counterfactual Validation

[View this lab on the QED Labs website](https://defenceeconomist.github.io/qedlabs/labs/interrupted-time-series-counterfactual-validation-lab.html)

## How To Use This Page

Use this as the third interrupted time-series lab.

- Assume no credible comparison area is available.
- Select the untreated forecasting backbone without using post-rollout outcomes.
- Validate every candidate on identical forecast origins and policy-relevant horizons.
- Report a primary and a near-performing comparator rather than hiding disagreement.
- Stop primary inference when a contextual predictor leaves its training support.


The synthetic repair-service case is regenerated offline from fixed seeds. The code never reads an external dataset or uses a post-rollout outcome during model selection.

## Training Goal

By the end of the lab, you should be able to:

1. screen contextual predictors by causal role, redundancy, coverage, and support;
2. distinguish in-sample fit from untreated forecast performance;
3. conduct expanding-window validation at 3-, 6-, and 12-month horizons;
4. select a model using an outcome-free normalized-RMSE rule;
5. report horizon-specific and cumulative observed-minus-expected gaps;
6. compare observed-exposure and simulated no-programme-exposure totals;
7. retain model disagreement and reject unsupported extrapolation; and
8. explain what an optional ARDL sensitivity asks—and what it cannot establish.

## Step 1: Generate The Common Synthetic Case

In [ ]:
required_packages <- c("ggplot2")
missing_packages <- required_packages[!vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_packages) > 0) {
  install.packages(missing_packages, repos = "https://cloud.r-project.org")
}
invisible(lapply(required_packages, library, character.only = TRUE))

simulate_repairs <- function(seed = 48127L) {
  set.seed(seed)
  n_months <- 132L
  implementation_time <- 88L
  time <- seq_len(n_months)
  date <- seq(as.Date("2014-01-01"), by = "month", length.out = n_months)
  implemented <- as.integer(time >= implementation_time)
  time_after <- pmax(0L, time - implementation_time)
  transition <- as.integer(time %in% implementation_time:(implementation_time + 2L))
  season_sin <- sin(2 * pi * time / 12)
  season_cos <- cos(2 * pi * time / 12)

  demand_pressure <- 100 + 0.10 * time + 5 * season_sin +
    as.numeric(arima.sim(list(ar = 0.45), n = n_months, sd = 1.3))
  demand_challenger <- demand_pressure + rnorm(n_months, 0, 0.7)
  weather_pressure <- 50 + 6 * season_cos +
    as.numeric(arima.sim(list(ar = 0.25), n = n_months, sd = 1.2))
  service_capacity <- 100 + 30 * implemented + rnorm(n_months, 0, 1)

  # A later extreme period is intentionally outside pre-rollout support.
  demand_pressure[time >= 121L] <- demand_pressure[time >= 121L] + 34
  demand_challenger[time >= 121L] <- demand_challenger[time >= 121L] + 34

  service_noise <- as.numeric(arima.sim(list(ar = c(0.48, 0.18)), n = n_months, sd = 8))
  programme_effect <- 68 * implemented + 0.9 * time_after - 22 * transition
  completed <- pmax(1L, round(
    735 + 0.28 * time + 19 * season_cos + 0.85 * (demand_pressure - 100) +
      programme_effect + service_noise
  ))
  withdrawn <- pmax(1L, round(72 + 5 * season_sin + rnorm(n_months, 0, 3)))
  transferred <- pmax(1L, round(28 + rnorm(n_months, 0, 2)))
  closures <- completed + withdrawn + transferred
  new_requests <- pmax(1L, round(
    875 + 0.55 * time + 17 * season_sin + 1.2 * (demand_pressure - 100) +
      rnorm(n_months, 0, 8)
  ))
  scope_change <- ifelse(time == implementation_time, -620L, 0L)

  open_cases <- integer(n_months)
  open_cases[[1]] <- 4400L
  for (index in 2:n_months) {
    open_cases[[index]] <- open_cases[[index - 1L]] +
      new_requests[[index]] - closures[[index]] + scope_change[[index]]
  }

  lag_open_cases <- c(NA, head(open_cases, -1L))
  closure_rate <- 1000 * closures / lag_open_cases

  data.frame(
    time,
    date,
    implemented,
    time_after,
    transition,
    season_sin,
    season_cos,
    demand_pressure,
    demand_challenger,
    weather_pressure,
    service_capacity,
    completed,
    withdrawn,
    transferred,
    closures,
    new_requests,
    scope_change,
    open_cases,
    lag_open_cases,
    closure_rate
  )
}

repairs <- simulate_repairs()
implementation_time <- min(repairs$time[repairs$implemented == 1L])
training_data <- subset(repairs, time < implementation_time & is.finite(closure_rate))
primary_impact <- subset(repairs, time >= implementation_time & time <= implementation_time + 11L)
unsupported_extension <- subset(repairs, time >= 121L)

stopifnot(
  max(training_data$time) < implementation_time,
  min(primary_impact$time) == implementation_time,
  nrow(primary_impact) == 12L,
  all(repairs$closures == repairs$completed + repairs$withdrawn + repairs$transferred),
  all(repairs$open_cases[-1L] == repairs$open_cases[-nrow(repairs)] +
    repairs$new_requests[-1L] - repairs$closures[-1L] + repairs$scope_change[-1L])
)

## Step 2: Screen Contextual Predictors Before Validation

In [ ]:
predictor_roles <- data.frame(
  Predictor = c(
    "Demand pressure",
    "Demand challenger",
    "Weather pressure",
    "Service capacity"
  ),
  Causal_role = c(
    "Independent demand pressure and untreated predictor",
    "Alternative measure of the same demand mechanism",
    "Seasonal external demand pressure",
    "Changed by the programme"
  ),
  Decision = c(
    "Primary contextual candidate",
    "Redundancy diagnostic only",
    "Single-proxy sensitivity",
    "Exclude: downstream of rollout"
  )
)

predictor_roles

predictor_correlations <- cor(training_data[c(
  "demand_pressure",
  "demand_challenger",
  "weather_pressure"
)])

predictor_correlations

stopifnot(
  predictor_correlations["demand_pressure", "demand_challenger"] > 0.9,
  predictor_roles$Decision[predictor_roles$Predictor == "Service capacity"] ==
    "Exclude: downstream of rollout"
)

The two demand measures are alternatives. Entering both would ask highly correlated indicators to divide the same mechanism. Service capacity is excluded because rollout changes it.

## Step 3: Define Candidate Untreated Models

The forecasting competition contains four deliberately different backbones:

1. seasonal naive;
2. linear trend plus annual Fourier terms;
3. AR(2) persistence plus annual Fourier terms; and
4. trend, annual Fourier terms, and the theory-led demand-pressure candidate.

In [ ]:
candidate_models <- c(
  "Seasonal naive",
  "Trend + season",
  "AR(2) + season",
  "Demand + trend"
)

forecast_candidate <- function(model_name, training, future) {
  if (model_name == "Seasonal naive") {
    lookup <- setNames(training$closure_rate, training$time)
    prediction <- unname(lookup[as.character(future$time - 12L)])
  } else if (model_name == "Trend + season") {
    fit <- lm(
      closure_rate ~ time + season_sin + season_cos,
      data = training
    )
    prediction <- as.numeric(predict(fit, newdata = future))
  } else if (model_name == "Demand + trend") {
    fit <- lm(
      closure_rate ~ time + season_sin + season_cos + demand_pressure,
      data = training
    )
    prediction <- as.numeric(predict(fit, newdata = future))
  } else if (model_name == "AR(2) + season") {
    training_xreg <- cbind(
      time = training$time,
      season_sin = training$season_sin,
      season_cos = training$season_cos
    )
    future_xreg <- cbind(
      time = future$time,
      season_sin = future$season_sin,
      season_cos = future$season_cos
    )
    fit <- arima(
      training$closure_rate,
      order = c(2, 0, 0),
      xreg = training_xreg,
      method = "ML"
    )
    prediction <- as.numeric(predict(
      fit,
      n.ahead = nrow(future),
      newxreg = future_xreg
    )$pred)
  } else {
    stop("Unknown candidate model")
  }

  stopifnot(length(prediction) == nrow(future), all(is.finite(prediction)))
  prediction
}

## Step 4: Validate On Identical Pre-Rollout Origins

Every model uses the same five expanding-window origins. At each origin, it forecasts the next twelve untreated months. Errors are summarized at 3, 6, and 12 months.

In [ ]:
validation_origins <- c(51L, 57L, 63L, 69L, 75L)
validation_horizons <- c(3L, 6L, 12L)

validation_errors <- do.call(rbind, lapply(candidate_models, function(model_name) {
  do.call(rbind, lapply(validation_origins, function(origin) {
    training <- subset(training_data, time <= origin)
    future <- subset(training_data, time > origin & time <= origin + 12L)
    predictions <- forecast_candidate(model_name, training, future)

    do.call(rbind, lapply(validation_horizons, function(horizon) {
      row <- which(future$time == origin + horizon)
      data.frame(
        Model = model_name,
        Origin = origin,
        Horizon = horizon,
        Error = future$closure_rate[row] - predictions[row],
        Training_end = max(training$time),
        Forecast_time = future$time[row]
      )
    }))
  }))
}))

stopifnot(
  all(validation_errors$Training_end < validation_errors$Forecast_time),
  max(validation_errors$Forecast_time) < implementation_time,
  setequal(unique(validation_errors$Origin), validation_origins),
  all(table(validation_errors$Model, validation_errors$Horizon) == length(validation_origins))
)

validation_scores <- aggregate(
  Error ~ Model + Horizon,
  data = validation_errors,
  FUN = function(values) sqrt(mean(values^2))
)
names(validation_scores)[names(validation_scores) == "Error"] <- "RMSE"

naive_rmse <- validation_scores[validation_scores$Model == "Seasonal naive", c("Horizon", "RMSE")]
names(naive_rmse)[2] <- "Naive_RMSE"
validation_scores <- merge(validation_scores, naive_rmse, by = "Horizon")
validation_scores$Normalized_RMSE <- validation_scores$RMSE / validation_scores$Naive_RMSE

overall_scores <- aggregate(
  Normalized_RMSE ~ Model,
  data = validation_scores,
  FUN = mean
)
overall_scores <- overall_scores[order(overall_scores$Normalized_RMSE), ]
row.names(overall_scores) <- NULL

validation_scores[order(validation_scores$Horizon, validation_scores$Normalized_RMSE), ]
overall_scores

The equal-weight mean of the three normalized RMSE values is fixed in advance. It prevents the largest-scale horizon from dominating and keeps the seasonal-naive benchmark visible.

## Step 5: Select Primary And Comparator Models

In [ ]:
selected_model <- overall_scores$Model[[1L]]
comparator_model <- overall_scores$Model[[2L]]

selection_record <- data.frame(
  Role = c("Primary", "Near-performing comparator"),
  Model = c(selected_model, comparator_model),
  Mean_normalized_RMSE = c(
    overall_scores$Normalized_RMSE[[1L]],
    overall_scores$Normalized_RMSE[[2L]]
  )
)

selection_record

stopifnot(
  selected_model %in% candidate_models,
  comparator_model %in% candidate_models,
  selected_model != comparator_model,
  selection_record$Mean_normalized_RMSE[[1L]] <= selection_record$Mean_normalized_RMSE[[2L]]
)

Do not inspect any post-rollout estimate before accepting this record. "Primary" means selected by the forecasting rule, not causally proven.

## Step 6: Forecast The Primary Impact Window

Use pre-rollout validation RMSE to construct horizon-dependent model-based prediction ranges. This keeps the same empirical forecasting scale in selection and reporting.

In [ ]:
forecast_with_intervals <- function(model_name, training, future) {
  prediction <- forecast_candidate(model_name, training, future)
  model_validation <- subset(validation_scores, Model == model_name)
  rmse_by_month <- approx(
    x = model_validation$Horizon,
    y = model_validation$RMSE,
    xout = seq_len(nrow(future)),
    method = "linear",
    rule = 2
  )$y

  data.frame(
    time = future$time,
    date = future$date,
    observed = future$closure_rate,
    expected = prediction,
    prediction_se = rmse_by_month,
    lower_95 = prediction - qnorm(0.975) * rmse_by_month,
    upper_95 = prediction + qnorm(0.975) * rmse_by_month,
    gap = future$closure_rate - prediction,
    lag_open_cases = future$lag_open_cases,
    new_requests = future$new_requests
  )
}

primary_forecast <- forecast_with_intervals(selected_model, training_data, primary_impact)
comparator_forecast <- forecast_with_intervals(comparator_model, training_data, primary_impact)

horizon_rows <- c(1L, 6L, 12L)
horizon_report <- rbind(
  transform(primary_forecast[horizon_rows, ], Model = selected_model),
  transform(comparator_forecast[horizon_rows, ], Model = comparator_model)
)
horizon_report$Horizon <- rep(c("Immediate", "6 months", "12 months"), 2)
horizon_report <- horizon_report[c(
  "Model",
  "Horizon",
  "observed",
  "expected",
  "gap",
  "lower_95",
  "upper_95"
)]

horizon_report

stopifnot(
  max(training_data$time) < min(primary_impact$time),
  all(is.finite(as.matrix(horizon_report[c("observed", "expected", "gap", "lower_95", "upper_95")]))),
  all(horizon_report$upper_95 > horizon_report$lower_95)
)

The interval is a model-based forecast range. It is not a design-based causal confidence interval.

## Step 7: Plot Observed And Counterfactual Paths

In [ ]:
forecast_plot_data <- rbind(
  data.frame(date = primary_forecast$date, series = paste("Expected:", selected_model), value = primary_forecast$expected),
  data.frame(date = comparator_forecast$date, series = paste("Expected:", comparator_model), value = comparator_forecast$expected)
)

ggplot() +
  geom_line(
    data = subset(repairs, time >= implementation_time - 24L & time <= implementation_time + 11L),
    aes(date, closure_rate),
    colour = "#4b615b",
    linewidth = 0.7
  ) +
  geom_ribbon(
    data = primary_forecast,
    aes(date, ymin = lower_95, ymax = upper_95),
    fill = "#24527a",
    alpha = 0.15
  ) +
  geom_line(
    data = forecast_plot_data,
    aes(date, value, colour = series, linetype = series),
    linewidth = 1
  ) +
  geom_vline(xintercept = repairs$date[implementation_time], linetype = "dashed", colour = "#c05a2a") +
  scale_colour_manual(values = c("#24527a", "#bf6b21")) +
  labs(x = NULL, y = "Closures per 1,000 open cases", colour = NULL, linetype = NULL) +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom")

Where the two paths diverge, the estimated programme gap depends on whether untreated dynamics are represented mainly by trend, persistence, or context.

## Step 8: Convert Rate Gaps To Cumulative Closures

The primary total applies the expected rate to the stock actually exposed. The sensitivity lets the no-programme stock evolve recursively.

In [ ]:
primary_forecast$expected_closures_observed_exposure <-
  primary_forecast$expected / 1000 * primary_forecast$lag_open_cases
primary_forecast$observed_excess_closures <-
  primary_impact$closures - primary_forecast$expected_closures_observed_exposure

counterfactual_open <- numeric(nrow(primary_forecast))
counterfactual_lag <- numeric(nrow(primary_forecast))
counterfactual_expected_closures <- numeric(nrow(primary_forecast))

counterfactual_lag[[1L]] <- repairs$open_cases[implementation_time - 1L]
for (index in seq_len(nrow(primary_forecast))) {
  if (index > 1L) {
    counterfactual_lag[[index]] <- counterfactual_open[[index - 1L]]
  }
  counterfactual_expected_closures[[index]] <-
    primary_forecast$expected[[index]] / 1000 * counterfactual_lag[[index]]
  counterfactual_open[[index]] <- counterfactual_lag[[index]] +
    primary_forecast$new_requests[[index]] - counterfactual_expected_closures[[index]]
}

primary_forecast$counterfactual_lag_open <- counterfactual_lag
primary_forecast$expected_closures_counterfactual_exposure <- counterfactual_expected_closures
primary_forecast$counterfactual_excess_closures <-
  primary_impact$closures - counterfactual_expected_closures

cumulative_report <- data.frame(
  Exposure = c("Observed stock at risk", "Simulated no-programme stock"),
  Excess_closures = c(
    sum(primary_forecast$observed_excess_closures),
    sum(primary_forecast$counterfactual_excess_closures)
  )
)

cumulative_report

stopifnot(
  all(counterfactual_open > 0),
  all(is.finite(cumulative_report$Excess_closures))
)

Explain why these totals answer different questions. Do not call the inferred difference "observed additional closures."

## Step 9: Audit Predictor Support And Stop The Extension

In [ ]:
training_range <- range(training_data$demand_pressure)
primary_range <- range(primary_impact$demand_pressure)
extension_range <- range(unsupported_extension$demand_pressure)

support_audit <- data.frame(
  Period = c("Pre-rollout training", "Primary 12-month impact", "Later extension"),
  Minimum = c(training_range[[1L]], primary_range[[1L]], extension_range[[1L]]),
  Maximum = c(training_range[[2L]], primary_range[[2L]], extension_range[[2L]]),
  Outside_training = c(
    FALSE,
    any(primary_impact$demand_pressure < training_range[[1L]] |
      primary_impact$demand_pressure > training_range[[2L]]),
    any(unsupported_extension$demand_pressure < training_range[[1L]] |
      unsupported_extension$demand_pressure > training_range[[2L]])
  )
)

support_audit

stopifnot(
  !support_audit$Outside_training[support_audit$Period == "Primary 12-month impact"],
  support_audit$Outside_training[support_audit$Period == "Later extension"]
)

The primary report ends after twelve months. The later period is retained only to demonstrate support failure; it is not an impact estimate.

## Step 10: Write The Counterfactual Report

Use these headings:

### Outcome And Exposure

Define the closure rate, lagged open-case exposure, and cumulative closure count.

### Predictor Decisions

Explain why demand pressure entered the competition, its challenger was treated as redundant, weather was reserved as a sensitivity, and service capacity was excluded.

### Validation Rule

List the five origins, three horizons, seasonal-naive benchmark, normalized-RMSE rule, primary model, and comparator.

### Short-Horizon Results

Report immediate, six-month, twelve-month, and cumulative gaps for both models. Distinguish the prediction interval from causal uncertainty.

### Model Dependence

State which conclusions are shared and where the paths diverge.

### Support And Stopping Rule

Explain why the later extension is rejected.

## Optional Appendix: ARDL And Cointegration Sensitivity

This exercise asks whether the closure rate and demand pressure share a stable long-run level relationship with short-run adjustment. It does **not** estimate programme attribution.

In [ ]:
ardl_data <- subset(repairs, time < implementation_time & is.finite(closure_rate))
ardl_data$delta_y <- c(NA, diff(ardl_data$closure_rate))
ardl_data$delta_x <- c(NA, diff(ardl_data$demand_pressure))
ardl_data$lag_y <- c(NA, head(ardl_data$closure_rate, -1L))
ardl_data$lag_x <- c(NA, head(ardl_data$demand_pressure, -1L))

unrestricted_ecm <- lm(
  delta_y ~ lag_y + lag_x + delta_x + time + season_sin + season_cos,
  data = ardl_data
)
restricted_ecm <- lm(
  delta_y ~ delta_x + time + season_sin + season_cos,
  data = ardl_data
)

bounds_style_test <- anova(restricted_ecm, unrestricted_ecm)
ecm_summary <- data.frame(
  Quantity = c("Bounds-style F statistic", "Lagged outcome coefficient", "Lagged context coefficient"),
  Value = c(
    bounds_style_test$F[[2L]],
    coef(unrestricted_ecm)[["lag_y"]],
    coef(unrestricted_ecm)[["lag_x"]]
  )
)

ecm_summary

stopifnot(all(is.finite(ecm_summary$Value)))

Before interpreting an ARDL bounds test:

1. audit integration order and rule out any I(2) series;
2. select lag orders on a common sample;
3. use critical bounds appropriate to the deterministic terms and number of level regressors;
4. label a statistic between the bounds inconclusive; and
5. keep the result separate from the programme-effect estimate [@pesaran2001; @natsiopoulos2022].

## Optional Appendix: Selection-Aware Uncertainty

A fuller model-based interval would repeat this workflow inside each resample:

1. resample pre-rollout residual blocks;
2. reconstruct an untreated series;
3. rerun the full expanding-window competition;
4. reselect the primary model;
5. recursively forecast twelve months; and
6. summarize the distribution of horizon and cumulative gaps.

This propagates parameter and selection uncertainty. It still does not simulate undocumented confounding, an unknown boundary change, or a treated-only measurement break.

## Suggested Close

The primary counterfactual is the model that best predicts held-out untreated observations at the horizons the evaluation will report. A comparator keeps model dependence visible. Neither forecasting success nor an advanced dynamic model substitutes for a credible causal design.